# REINFORCE — learn a policy by trial and error

**Agents & RL** · the simplest policy-gradient method (the root of PPO, used in RLHF).

An agent explores a gridworld; we reward reaching the goal and push up the probability of actions that led to high return. Self-contained — no gym needed.

> CPU is fine.

In [ ]:
import os, torch, torch.nn as nn, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 600)); N = 5; GOAL = (N - 1, N - 1); GAMMA = 0.95

## 1 · The environment (a N×N gridworld)

In [ ]:
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]
def step(pos, a):
    x = min(max(pos[0] + MOVES[a][0], 0), N - 1); y = min(max(pos[1] + MOVES[a][1], 0), N - 1)
    done = (x, y) == GOAL
    return (x, y), (5.0 if done else -0.1), done
def feat(pos):
    v = torch.zeros(2 * N, device=device); v[pos[0]] = 1.0; v[N + pos[1]] = 1.0; return v

## 2 · Policy network + REINFORCE update

In [ ]:
policy = nn.Sequential(nn.Linear(2 * N, 64), nn.ReLU(), nn.Linear(64, 4)).to(device)
value = nn.Sequential(nn.Linear(2 * N, 64), nn.ReLU(), nn.Linear(64, 1)).to(device)   # critic baseline
opt = torch.optim.Adam(list(policy.parameters()) + list(value.parameters()), 2e-3)
def episode():
    pos = (torch.randint(0, N, (1,)).item(), torch.randint(0, N, (1,)).item())
    states, logps, ents, rews = [], [], [], []
    for _ in range(40):
        if pos == GOAL: break
        f = feat(pos); d = torch.distributions.Categorical(logits=policy(f)); a = d.sample()
        states.append(f); logps.append(d.log_prob(a)); ents.append(d.entropy())
        pos, r, done = step(pos, a.item()); rews.append(r)
        if done: break
    return states, logps, ents, rews

## 3 · Train — push up high-return actions

In [ ]:
hist = []
for step_i in range(STEPS + 1):
    batch_loss, rets = [], []
    for _ in range(16):
        states, logps, ents, rews = episode()
        if not logps: continue
        G = 0; returns = []
        for r in reversed(rews): G = r + GAMMA * G; returns.insert(0, G)
        returns = torch.tensor(returns, device=device)
        V = value(torch.stack(states)).squeeze(-1)                 # state-value baseline
        adv = (returns - V).detach(); adv = (adv - adv.mean()) / (adv.std(unbiased=False) + 1e-6)
        pg = -(torch.stack(logps) * adv).sum()                     # policy gradient with advantage
        vloss = ((V - returns) ** 2).mean()                        # critic regression
        ent = -0.01 * torch.stack(ents).sum()                      # entropy bonus (exploration)
        batch_loss.append(pg + 0.5 * vloss + ent); rets.append(sum(rews))
    loss = torch.stack(batch_loss).mean(); opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(list(policy.parameters()) + list(value.parameters()), 1.0); opt.step()
    if step_i % max(1, STEPS // 10) == 0:
        ar = sum(rets) / len(rets); hist.append((step_i, round(ar, 2))); print(f"step {step_i:4d}  avg return {ar:6.2f}")

## 4 · Compare — return climbs as the policy learns

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(*zip(*hist), "-o"); ax.set_xlabel("update"); ax.set_ylabel("avg episode return"); ax.grid(alpha=.3); ax.set_title("REINFORCE learns to reach the goal")
plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/AG_reinforce_gridworld/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/AG_reinforce_gridworld"; os.makedirs(run, exist_ok=True)
torch.save(policy.state_dict(), f"{run}/policy.pt")
json.dump({"return": hist}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/ropedia-" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
Policy gradients scale up to **D · Scene / world** embodied control.

### Where to go next
- Add a value baseline → **Actor-Critic**, then clipped updates → **PPO** (the algorithm behind RLHF).
- Swap the gridworld for a Gym/MuJoCo task; learn from pixels with a world model (track D).